# LAB 3 – Bài 4 (Trang 9, câu c → hết): Neural Network phân lớp Iris bằng k-fold cross validation
Các câu a, b (đọc và chuẩn hóa dữ liệu) được thực hiện lại ở đầu bài để notebook chạy độc lập.

In [9]:
HO_TEN = "Trang Kim Đạt"
MSSV = "6451071018"
print(f"{HO_TEN} – {MSSV}")

Trang Kim Đạt – 6451071018


In [10]:
print(f"{HO_TEN} – {MSSV}")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

np.random.seed(42)
tf.keras.utils.set_random_seed(42)

Trang Kim Đạt – 6451071018


### Câu a, b (chuẩn bị dữ liệu)

In [11]:
print(f"{HO_TEN} – {MSSV}")
iris = load_iris()
X, Y = iris.data, iris.target
X_max = X.max(axis=0)
X = X / X_max              # chuẩn hóa về [0, 1]
so_lop = len(iris.target_names)
print(X.shape, Y.shape, "| max sau chuẩn hóa:", X.max(axis=0))

Trang Kim Đạt – 6451071018
(150, 4) (150,) | max sau chuẩn hóa: [1. 1. 1. 1.]


## Câu c) Hàm `create_model` (giống câu 1.h)

In [12]:
print(f"{HO_TEN} – {MSSV}")
def create_model():
    model = Sequential([
        Input(shape=(4,)),
        Dense(10, activation="relu"),
        Dense(20, activation="relu"),
        Dense(so_lop, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

create_model().summary()

Trang Kim Đạt – 6451071018


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_33 (Dense)                │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

## Câu d) Huấn luyện 10-fold cross validation
Mỗi fold: huấn luyện 100 epoch, batch_size = 1 (giống câu 1.h); tính accuracy, precision, recall, F1-score (macro) trên tập validation của fold và lưu lại mô hình, đồng thời lưu P/R/F1 của từng lớp để dùng cho câu f.

In [13]:
print(f"{HO_TEN} – {MSSV}")
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
labels = list(range(so_lop))
models = []
acc_list, prec_list, rec_list, f1_list = [], [], [], []
prec_cls, rec_cls, f1_cls = [], [], []

for fold, (train_index, val_index) in enumerate(kfold.split(X), start=1):
    model = create_model()
    model.fit(X[train_index], Y[train_index], epochs=100, batch_size=1, verbose=0)
    y_val = Y[val_index]
    y_pred = np.argmax(model.predict(X[val_index], verbose=0), axis=1)

    acc = accuracy_score(y_val, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_val, y_pred, average="macro", zero_division=0)
    p_c, r_c, f_c, _ = precision_recall_fscore_support(y_val, y_pred, labels=labels, average=None, zero_division=0)

    models.append(model)
    acc_list.append(acc); prec_list.append(p); rec_list.append(r); f1_list.append(f)
    prec_cls.append(p_c); rec_cls.append(r_c); f1_cls.append(f_c)
    print(f"Fold {fold:2d}: accuracy = {acc:.4f} | precision = {p:.4f} | recall = {r:.4f} | f1-score = {f:.4f}")

Trang Kim Đạt – 6451071018
Fold  1: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000
Fold  2: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000
Fold  3: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000
Fold  4: accuracy = 0.9333 | precision = 0.9524 | recall = 0.9333 | f1-score = 0.9373
Fold  5: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000
Fold  6: accuracy = 0.9333 | precision = 0.9167 | recall = 0.9444 | f1-score = 0.9221
Fold  7: accuracy = 0.8667 | precision = 0.9167 | recall = 0.8333 | f1-score = 0.8413
Fold  8: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000
Fold  9: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000
Fold 10: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000


## Câu e) Kết quả trung bình của mô hình sau 10-fold

In [14]:
print(f"{HO_TEN} – {MSSV}")
print(f"Average Accuracy : {np.mean(acc_list):.4f}")
print(f"Average Recall   : {np.mean(rec_list):.4f}")
print(f"Average Precision: {np.mean(prec_list):.4f}")
print(f"Average F1-score : {np.mean(f1_list):.4f}")

Trang Kim Đạt – 6451071018
Average Accuracy : 0.9733
Average Recall   : 0.9711
Average Precision: 0.9786
Average F1-score : 0.9701


## Câu f) Average Recall, Precision, F1-score cho từng lớp

In [15]:
print(f"{HO_TEN} – {MSSV}")
bang = pd.DataFrame({
    "Average Precision": np.mean(prec_cls, axis=0),
    "Average Recall": np.mean(rec_cls, axis=0),
    "Average F1-score": np.mean(f1_cls, axis=0),
}, index=iris.target_names).round(4)
bang

Trang Kim Đạt – 6451071018


,Average Precision,Average Recall,Average F1-score
setosa,1.0000,1.0000,1.0000
versicolor,0.9857,0.9333,0.9499
virginica,0.9500,0.9800,0.9603


## Câu g) Dự đoán 03 mẫu mới bằng các mô hình có F1-score tốt nhất
Chọn tất cả các fold đạt F1-score lớn nhất (có thể có nhiều fold cùng điểm).

In [16]:
print(f"{HO_TEN} – {MSSV}")
mau_moi = np.array([[6.2, 2.9, 4.3, 1.3],
                    [5.1, 3.5, 1.4, 0.2],
                    [7.3, 2.8, 6.4, 2.1]])
mau_moi_chuan_hoa = mau_moi / X_max

f1_max = np.max(f1_list)
fold_tot_nhat = [i for i, f in enumerate(f1_list) if np.isclose(f, f1_max)]
print(f"F1-score tốt nhất = {f1_max:.4f} tại các fold:", [i + 1 for i in fold_tot_nhat])

ket_qua = {}
for i in fold_tot_nhat:
    pred = np.argmax(models[i].predict(mau_moi_chuan_hoa, verbose=0), axis=1)
    ket_qua[f"Mô hình fold {i + 1}"] = iris.target_names[pred]
kq = pd.DataFrame(ket_qua)
kq.insert(0, "Mẫu", [str(m) for m in mau_moi.tolist()])
kq

Trang Kim Đạt – 6451071018
F1-score tốt nhất = 1.0000 tại các fold: [1, 2, 3, 5, 8, 9, 10]


,Mẫu,Mô hình fold 1,Mô hình fold 2,Mô hình fold 3,Mô hình fold 5,Mô hình fold 8,Mô hình fold 9,Mô hình fold 10
0,"[6.2, 2.9, 4.3, 1.3]",versicolor,versicolor,versicolor,versicolor,versicolor,versicolor,versicolor
1,"[5.1, 3.5, 1.4, 0.2]",setosa,setosa,setosa,setosa,setosa,setosa,setosa
2,"[7.3, 2.8, 6.4, 2.1]",virginica,virginica,virginica,virginica,virginica,virginica,virginica
